# 信用卡客戶流失分析

> **結論**：交易越來越少的客戶最容易流失。只要每月追蹤「交易次數連續下滑」的客戶並主動挽留，就能在客戶真正流失前介入。

**情境**：假設我是某銀行信用卡部門的初階數據分析師。公司問了三個問題——

1. 我們現在的流失率有多高？
2. 會流失的客戶有什麼共同特徵？
3. 有限的行銷資源，該優先留住誰？

**Notebook 怎麼做**：用 **Python（pandas / scikit-learn）** 一條龍完成——載入、品質檢查、清洗、探索式分析(EDA)、建模、結論。
---
**資料來源**：Kaggle — Credit Card customers（`BankChurners.csv`，約 10,000 筆）。
**怎麼跑**：在 Google Colab 左側「檔案」上傳 `BankChurners.csv`，再由上往下執行每個 cell。

## 1. 環境設定與載入原始資料

用 pandas 直接讀取 CSV。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# 載入原始資料
raw = pd.read_csv("BankChurners.csv")

print("原始資料大小（列, 欄）:", raw.shape)
raw.head()

## 2. 資料品質檢查

清洗前先看一眼資料狀況：筆數、整體流失率、缺漏值、重複客戶。

In [ ]:
# 筆數 + 整體流失率
n_rows = len(raw)
churn_rate = (raw["Attrition_Flag"] == "Attrited Customer").mean()

print(f"資料筆數: {n_rows}")
print(f"整體流失率: {churn_rate:.3f}")

In [ ]:
# 缺漏值：檢查幾個重要的類別欄位
raw[["Income_Category", "Education_Level", "Marital_Status"]].isna().sum()

In [ ]:
# 重複客戶：同一個 CLIENTNUM 是否出現多次
dup_count = raw["CLIENTNUM"].duplicated().sum()
print(f"重複的 CLIENTNUM 數量: {dup_count}")  # 0 = 沒有重複

**檢查結果**：約 10,000 筆、流失率約 16%、沒有缺漏值也沒有重複客戶。
（資料本身乾淨，但部分文字欄位含 `Unknown`，我們把它當成一個獨立類別保留，這是誠實處理真實資料的做法。）

## 3. 資料清洗（用 pandas）

做四件事：

1. **只保留有意義的欄位** — 刻意不選 `CLIENTNUM` 和兩個 `Naive_Bayes_Classifier` 欄位。後者是事先算好的預測結果，留著會造成**資料洩漏**，。
2. **刪掉 `Avg_Open_To_Buy`** — 這欄等於 `Credit_Limit − Total_Revolving_Bal`（實測兩者完全相等），和 `Credit_Limit` 相關係數高達 0.996。留著會造成**多重共線性**：迴歸會把它們的係數隨機拆分、正負不穩，使第 6 段的「流失因子解讀」失真。刪掉零資訊損失，又讓係數變得可信。
3. **去除文字欄位的多餘空白**（`str.strip()`），避免分組時把 `"Male"` 和 `"Male "` 當成兩組。
4. **建立目標欄位 `Churn`**（流失=1、留存=0）。

In [ ]:
# 1) 去除文字欄位的多餘空白
text_cols = ["Gender", "Education_Level", "Marital_Status",
             "Income_Category", "Card_Category", "Attrition_Flag"]
for c in text_cols:
    raw[c] = raw[c].astype(str).str.strip()

# 2) 只保留有意義的欄位
#    - 不選 CLIENTNUM 與兩個 Naive_Bayes_Classifier（資料洩漏）
#    - 不選 Avg_Open_To_Buy（與 Credit_Limit 共線，r=0.996）
keep_cols = [
    "Customer_Age", "Gender", "Dependent_count", "Education_Level", "Marital_Status",
    "Income_Category", "Card_Category", "Months_on_book", "Total_Relationship_Count",
    "Months_Inactive_12_mon", "Contacts_Count_12_mon", "Credit_Limit", "Total_Revolving_Bal",
    "Total_Amt_Chng_Q4_Q1", "Total_Trans_Amt", "Total_Trans_Ct", "Total_Ct_Chng_Q4_Q1",
    "Avg_Utilization_Ratio", "Attrition_Flag",
]
df = raw[keep_cols].copy()

# 3) 建立目標欄位 Churn（流失=1、留存=0）
df["Churn"] = (df["Attrition_Flag"] == "Attrited Customer").astype(int)

print("清洗後資料大小（列, 欄）:", df.shape)
df.head()

## 4. 探索性分析 (EDA)

資料乾淨了，先比較「流失」與「留存」兩群客戶的平均行為。

In [ ]:
behavior_cols = ["Total_Trans_Ct", "Total_Trans_Amt", "Avg_Utilization_Ratio",
                 "Months_Inactive_12_mon", "Contacts_Count_12_mon", "Total_Ct_Chng_Q4_Q1"]

df.groupby("Attrition_Flag")[behavior_cols].mean().round(2)

**觀察**：流失客戶的交易次數、交易金額、額度使用率都明顯較低，不活躍月數與客服聯絡次數較高。
這透露一個故事——**客戶通常是先變得不活躍，然後才流失**。

### 4-1 哪些客群的流失率最高？

In [ ]:
rate_card = df.groupby("Card_Category")["Churn"].mean().sort_values(ascending=False) * 100

plt.figure(figsize=(7, 4))
rate_card.plot(kind="bar", color="#c0392b")
plt.axhline(df["Churn"].mean()*100, color="gray", ls="--", label="Overall average")
plt.title("Churn Rate by Card Category")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=20)
plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
rate_income = df.groupby("Income_Category")["Churn"].mean().sort_values(ascending=False) * 100

plt.figure(figsize=(7, 4))
rate_income.plot(kind="bar", color="#c0392b")
plt.axhline(df["Churn"].mean()*100, color="gray", ls="--", label="Overall average")
plt.title("Churn Rate by Income Category")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=20)
plt.legend()
plt.tight_layout(); plt.show()

### 4-2 交易次數的分布差異（最關鍵的特徵）

In [ ]:
plt.figure(figsize=(8, 4))
sns.kdeplot(data=df, x="Total_Trans_Ct", hue="Attrition_Flag",
            fill=True, common_norm=False, alpha=0.4)
plt.title("Transaction Count: Churned vs Existing")
plt.xlabel("Total Transaction Count (last 12 months)")
plt.tight_layout(); plt.show()

**觀察**：留存客戶集中在交易次數較高的區間，流失客戶集中在偏低區間。交易次數幾乎可以當作「流失的早期警訊」。

### 4-3 哪些數值欄位和流失最相關？

In [ ]:
num_df = df.select_dtypes(exclude="object")
corr = num_df.corr()["Churn"].drop("Churn").sort_values()

plt.figure(figsize=(7, 6))
corr.plot(kind="barh", color=np.where(corr < 0, "#2980b9", "#c0392b"))
plt.title("Correlation with Churn")
plt.xlabel("Correlation coefficient")
plt.tight_layout(); plt.show()

## 5. 建立預測模型（邏輯迴歸）

**為什麼選邏輯迴歸？** 它的係數可以清楚解釋「每個因子如何影響流失」。

**三個處理重點**：
1. 文字欄位用 `pd.get_dummies()` 轉成 0/1。
2. **標準化放進 `Pipeline`，且只 fit 在訓練集上**。如果先對整份資料算平均與標準差再切分，等於偷用了測試集的資訊（資料洩漏）；用 `Pipeline` 能保證標準化只看訓練集，測試集完全不參與訓練。
3. 流失客戶只占約 16%（不平衡），加上 `class_weight="balanced"`，並特別看 **recall**——真正會流失的客戶抓到了幾成。

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# 1) 特徵 X 與目標 y
X = df.drop(columns=["Attrition_Flag", "Churn"])
y = df["Churn"]

# 2) 文字欄位 → 0/1 數字欄位
X = pd.get_dummies(X, drop_first=True)

# 3) 先切訓練 / 測試集（標準化稍後在 Pipeline 內進行，只會 fit 在訓練集）
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

# 4) Pipeline：標準化 + 邏輯迴歸。scaler 只 fit 在訓練集，避免資料洩漏
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, class_weight="balanced"))
model.fit(X_train, y_train)

# 5) 評估
pred  = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred, target_names=["Existing", "Churn"]))
print("ROC-AUC: {:.3f}".format(roc_auc_score(y_test, proba)))

**怎麼解讀**：
- 流失那列的 **recall** 約 0.8，代表抓得出約 8 成真正會流失的客戶。
- **ROC-AUC** 約 0.9，整體區分能力很強。

In [ ]:
cm = confusion_matrix(y_test, pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred Existing", "Pred Churn"],
            yticklabels=["True Existing", "True Churn"])
plt.title("Confusion Matrix")
plt.tight_layout(); plt.show()

## 6. 哪些因子最能解釋流失？
係數讀法：**負值** = 該因子越高、越**不會**流失；**正值** = 越高、越**容易**流失。

In [ ]:
# 從 Pipeline 取出邏輯迴歸那一步，再讀它的係數
logreg = model.named_steps["logisticregression"]
coefs = pd.Series(logreg.coef_[0], index=X.columns)
top10 = coefs[coefs.abs().sort_values(ascending=False).head(10).index].sort_values()

plt.figure(figsize=(8, 5))
top10.plot(kind="barh", color=np.where(top10 < 0, "#2980b9", "#c0392b"))
plt.axvline(0, color="black", lw=0.8)
plt.title("Top 10 Drivers of Churn")
plt.xlabel("Coefficient  (negative = less churn, positive = more churn)")
plt.tight_layout(); plt.show()

top10

## 7. 結論與建議

### 關鍵發現
1. **交易次數 (Total_Trans_Ct) 是最強的流失訊號** — 流失客戶平均只交易約 45 次，留存客戶約 69 次。
2. **不活躍月數、客服聯絡次數**偏高也和流失正相關。
3. **與銀行往來越深（持有產品數越多）的客戶越不容易流失。**
4. **收入與流失沒有單純的線性關係**：最高收入（$120K+，約 17.3%）與最低收入（<$40K，約 17.2%）兩端流失率都偏高，中間級距較低，呈現微弱的 U 型；但各級距差距其實不大（約 13.5%–17.3%），所以這只是值得後續追蹤的線索，**不足以單獨下定論**。

### 建議
- **建立每月「流失預警名單」**：鎖定「流失機率高 ＋ 交易次數連續下滑」的客戶，主動提供回饋或專屬優惠，在他們離開前介入。
- **深耕客戶關係**：對只持有單一產品的客戶推薦合適的第二項產品，提高黏著度。
- **檢視客訴熱點**：分析客服聯絡次數偏高客群的客訴內容，找出可系統性改善的問題。
- **守住高價值客戶**：為高收入但低使用率的客戶設計專屬權益，並持續觀察兩端收入族群的流失狀況。

